## table_correspondance_renvois

**Fichier(s) source :** `./data/bofip_stock_live_20260521.tgz` (stock du 21.05.2026) et `./data/inventaire_bofip_stock_live_20260521.xlsx` (inventaire produit par `profilage_bofip_consolide.ipynb`)

**Fichier(s) de sortie :** DataFrame en mémoire

**Description :** Table de correspondance des renvois (dc:relation) d'un document du stock BOFiP (cas 8347-PGP) : pour chaque renvoi, son type (requires/references), sa sorte, sa présence (inventaire et/ou archive) et son code BOI correspondant ; suivi d'un récapitulatif des orphelins et des ressources requises manquantes.

## Etape 1. Chemins et identifiant

In [1]:
import tarfile, os, glob, re

TGZ_PATH = r"./data/bofip_stock_live_20260521.tgz"
INVENTAIRE_PATH = r"./data/inventaire_bofip_stock_live_20260521.xlsx"
IDENTIFIANT = '8347-PGP'

BASE = r"./data"
def _chercher(motifs):
    res=[]
    if os.path.isdir(BASE):
        for m in motifs:
            res += glob.glob(os.path.join(BASE,'**',m), recursive=True)
    return res
if not os.path.exists(TGZ_PATH):
    c=[x for x in _chercher(['*.tgz']) if 'stock' in os.path.basename(x).lower()]
    if c: TGZ_PATH=c[0]
if not os.path.exists(INVENTAIRE_PATH):
    inv=_chercher(['inventaire*stock*.xlsx','inventaire*.xlsx'])
    if inv: INVENTAIRE_PATH=inv[0]
print('Archive    :', TGZ_PATH if os.path.exists(TGZ_PATH) else 'INTROUVABLE')
print('Inventaire :', INVENTAIRE_PATH if os.path.exists(INVENTAIRE_PATH) else 'INTROUVABLE')
print('Document   :', IDENTIFIANT)

Archive    : ./data/bofip_stock_live_20260521.tgz
Inventaire : ./data/inventaire_bofip_stock_live_20260521.xlsx
Document   : 8347-PGP


## Etape 2. Lire l'archive et l'inventaire

L'inventaire donne, pour chaque identifiant present, son code BOI (colonne boi_base). L'archive donne la liste des codes physiquement presents (documents, pieces jointes, images).

In [2]:
# Codes presents physiquement dans l'archive
CODES_ARCHIVE=set()
NOMS=[]
if os.path.exists(TGZ_PATH):
    with tarfile.open(TGZ_PATH,'r:gz') as tar:
        NOMS=[m.name for m in tar.getmembers() if m.isfile()]
    for n in NOMS:
        for s in re.split(r'[\\/]', n):
            CODES_ARCHIVE.add(s); CODES_ARCHIVE.add(s.split('.')[0])
print('Fichiers dans l\'archive :', len(NOMS))

# Inventaire : identifiant -> code BOI (boi_base) et serie
INV={}
if os.path.exists(INVENTAIRE_PATH):
    try:
        import pandas as pd
        df=pd.read_excel(INVENTAIRE_PATH, usecols=['identifiant','boi_base','serie'])
        for _,r in df.iterrows():
            INV[str(r['identifiant'])]={'boi':r['boi_base'],'serie':r['serie']}
    except Exception:
        import openpyxl
        wb=openpyxl.load_workbook(INVENTAIRE_PATH, data_only=True, read_only=True)
        ws=wb.active; ent=[c.value for c in next(ws.iter_rows(min_row=1,max_row=1))]
        ii,ib,isr=ent.index('identifiant'),ent.index('boi_base'),ent.index('serie')
        for row in ws.iter_rows(min_row=2, values_only=True):
            if row[ii] is not None:
                INV[str(row[ii])]={'boi':row[ib],'serie':row[isr]}
print('Identifiants dans l\'inventaire :', len(INV))

Fichiers dans l'archive : 14993
Identifiants dans l'inventaire : 6311


## Etape 3. Lire les renvois du document (avec leur type)

In [3]:
def appartient(nom):
    for s in re.split(r'[\\/]', nom):
        if s==IDENTIFIANT or s.split('.')[0]==IDENTIFIANT: return True
    return False
fichiers_doc=[n for n in NOMS if appartient(n)]
renvois=[]  # (type, sorte, cible)
xmls=[n for n in fichiers_doc if n.lower().endswith('.xml')]
if xmls:
    with tarfile.open(TGZ_PATH,'r:gz') as tar:
        xml=tar.extractfile(xmls[0]).read().decode('utf-8',errors='replace')
    for m in re.findall(r'<dc:relation([^>]*)>([^<]+)</dc:relation>', xml):
        t=re.search(r'type="([^"]+)"', m[0]); typ=t.group(1) if t else ''
        val=m[1].strip()
        sorte=val.split(':',1)[0] if ':' in val else 'Autre'
        cible=(val.split(':',1)[1] if ':' in val else val).split('#',1)[0]
        renvois.append((typ,sorte,cible))
print('Renvois lus :', len(renvois))

Renvois lus : 15


## Etape 4. Table de correspondance

Pour chaque renvoi : type, sorte, code cible, presence, et code BOI correspondant (si la cible est dans l'inventaire).

In [4]:
def localiser(cible):
    di = cible in INV
    da = cible in CODES_ARCHIVE
    if di and da: return 'inventaire + archive'
    if di: return 'inventaire'
    if da: return 'archive (piece jointe)'
    return 'ABSENT (orphelin)'

print(f"{'Type':10}{'Sorte':10}{'Code cible':14}{'Presence':22}{'Code BOI correspondant'}")
print('-'*92)
for typ,sorte,cible in renvois:
    pres=localiser(cible)
    boi = INV[cible]['boi'] if cible in INV else ''
    print(f'{typ:10}{sorte:10}{cible:14}{pres:22}{boi}')

Type      Sorte     Code cible    Presence              Code BOI correspondant
--------------------------------------------------------------------------------------------
referencesActualite 12500-PGP     ABSENT (orphelin)     
requires  Fichier   13093-PGP     archive (piece jointe)
referencesContenu   13690-PGP     ABSENT (orphelin)     
referencesContenu   13691-PGP     ABSENT (orphelin)     
requires  Fichier   13694-PGP     ABSENT (orphelin)     
requires  Fichier   13695-PGP     ABSENT (orphelin)     
requires  Fichier   13705-PGP     ABSENT (orphelin)     
requires  Fichier   13706-PGP     ABSENT (orphelin)     
requires  Fichier   14122-PGP     ABSENT (orphelin)     
referencesContenu   5176-PGP      inventaire + archive  BOI-CAD-MAJ-10-50
referencesContenu   5251-PGP      inventaire + archive  BOI-CAD-MAJ-10-10
referencesContenu   5261-PGP      inventaire + archive  BOI-ANNX-000390
referencesContenu   5266-PGP      inventaire + archive  BOI-ANNX-000391
referencesContenu   531

## Etape 5. Recapitulatif des orphelins par sorte et par type

In [5]:
orph=[(typ,sorte,cible) for (typ,sorte,cible) in renvois if localiser(cible).startswith('ABSENT')]
print('Orphelins (cible introuvable partout) :', len(orph))
for sorte in ['Actualite','Contenu','Fichier','Autre']:
    lst=[f'{c} ({t})' for (t,s,c) in orph if s==sorte]
    if lst:
        print(f'  {sorte} ({len(lst)}) : ' + ', '.join(lst))

req=[c for (t,s,c) in renvois if t=='requires']
req_orph=[c for (t,s,c) in orph if t=='requires']
print()
print(f'Ressources requises (requires) : {len(req)} declarees, {len(req_orph)} manquantes')
if req_orph:
    print('  Manquantes :', ', '.join(req_orph))

Orphelins (cible introuvable partout) : 8
  Actualite (1) : 12500-PGP (references)
  Contenu (2) : 13690-PGP (references), 13691-PGP (references)
  Fichier (5) : 13694-PGP (requires), 13695-PGP (requires), 13705-PGP (requires), 13706-PGP (requires), 14122-PGP (requires)

Ressources requises (requires) : 6 declarees, 5 manquantes
  Manquantes : 13694-PGP, 13695-PGP, 13705-PGP, 13706-PGP, 14122-PGP
